# 02 - Quantum Fourier Transform

**Question:** how much noise can a QFT take before we can no longer get our input back?

The QFT moves information from the qubits' **bit values** into their **phases**. For an input number $x$ on $n$ qubits:

$$QFT|x\rangle = \frac{1}{\sqrt{2^n}}\sum_{y=0}^{2^n-1} e^{2\pi i\,xy/2^n}|y\rangle$$

Every outcome $y$ has the same probability. Only the phase $e^{2\pi i xy/2^n}$ remembers $x$, and a measurement cannot see a phase. So we test a **round trip**: input, QFT, inverse QFT, measure. Without noise the input always comes back.

**Score:** share of shots that return the input. **Baseline:** $1/2^n$. **Pass:** score at least 50% of the ideal score.

**Bit order:** the rightmost character is qubit 0.

In [ ]:
import matplotlib.pyplot as plt
from qiskit import qasm2
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

from noisy_qubits.algorithms import qft
from noisy_qubits.noise import DEVICES, NOISE_TYPES, build_noise_model, slider_label

## 1. The three gates

- **H** puts a qubit into an equal mix of 0 and 1. Its old value becomes a phase of 0 or 180 degrees.
- **CP(angle)** turns the phase only when both qubits are 1. The angles get smaller: 90, 45, 22.5 degrees...
- **SWAP** reverses the qubit order at the end.

Here is the QFT on 3 qubits. Read it from the highest qubit down: H, then smaller and smaller rotations controlled by the qubits below.

In [ ]:
qft.qft_circuit(3).draw("mpl")

## 2. See the information move into the phases

All 8 outcomes below have the same probability. Different inputs differ only in the phase angles.

In [ ]:
import numpy as np
for state in ["000", "001", "100"]:
    out = Statevector.from_label(state).evolve(qft.qft_circuit(3))
    print(f"input {state}: probabilities {np.round(out.probabilities(), 3)}")
    print(f"           phases (deg) {np.round(np.degrees(np.angle(out.data))) % 360}")

For input `001` (the number 1) the phase goes up 45 degrees from one outcome to the next. For `100` (the number 4) it goes up 180 degrees. The input sets how fast the phase turns.

## 3. The round trip circuit

X gates prepare the input. The **barriers** matter: on a fake device Qiskit optimises the circuit, and without them it cancels gates between the QFT and its inverse. Cancelled gates get no noise, so the score would look too good.

In [ ]:
qft.build(4, "0110").draw("mpl", fold=40)

## 4. No noise: matches theory

In [ ]:
for n in range(2, 11):
    r = qft.run(n, 256, seed=1)
    print(f"n={n:2d} input={r.meta['input_state']:>10} score={r.score:.3f} baseline={r.baseline:.4f}")
    assert r.score == 1.0

## 5. Adding gate noise

In [ ]:
results = {s: qft.run(10, 1024, build_noise_model("uniform", s), seed=11) for s in (1, 10, 20, 50)}
for s, r in results.items():
    verdict = "PASS" if r.score >= 0.5 * r.ideal_score else "below the pass line"
    print(f"{slider_label('uniform', s):32s} score={r.score:.3f}  {verdict}")

In [ ]:
sliders = [1] + list(range(5, 101, 5))
scores = [qft.run(10, 1024, build_noise_model("uniform", s), seed=11).score for s in sliders]
plt.figure(figsize=(7, 4))
plt.plot(sliders, scores, marker="o", label="QFT round trip")
plt.axhline(0.5, ls=":", c="red", label="pass line")
plt.axhline(1 / 1024, ls="--", c="grey", label="classical baseline (1/1024)")
plt.xlabel("noise slider"); plt.ylabel("success rate"); plt.legend(); plt.show()

## 6. More qubits, more gates, lower score

The number of CP gates grows with about $n^2$, so larger circuits collect more errors.

In [ ]:
nm = build_noise_model("uniform", 20)
for n in range(2, 11):
    r = qft.run(n, 1024, nm, seed=11)
    print(f"n={n:2d}  cp gates={r.circuit.count_ops().get('cp', 0):3d}  score={r.score:.3f}")

## 7. Which noise types hurt?

A phase flip never changes a plain 0 or 1, but the QFT stores information in phases, so it hurts here.

In [ ]:
for t in NOISE_TYPES:
    r = qft.run(10, 1024, build_noise_model("uniform", 20, t), seed=11)
    print(f"{t:18s} {r.score:.3f}")

## 8. Fake IBM devices (slider 50 = the real device)

A real chip only connects neighbouring qubits, so Qiskit adds SWAPs, and each CP and SWAP becomes several CX gates. This is why scores here are far lower than in uniform mode, where a CP or SWAP counts as one gate. The two modes' sliders are not comparable.

In [ ]:
for d in DEVICES:
    r = qft.run(10, 1024, build_noise_model("device", 50, device=d), device=d, seed=11)
    print(f"{d:14s} {r.score:.3f}")

## 9. Experiment: approximate QFT

`approximation_degree = d` removes the d smallest rotation sizes. Fewer gates means less noise.

**A catch:** the round trip always returns the input without noise, because any circuit followed by its inverse does nothing. So the round-trip score cannot tell whether the approximation is still a QFT. We therefore also report the **fidelity**: how close the approximate QFT's output is to the exact one (1 = same, 0 = completely different). It is measured here for this one input.

In [ ]:
nm = build_noise_model("uniform", 20)
print(" d  cp gates  fidelity to exact QFT  round-trip score (slider 20)")
for d in (0, 3, 5, 7, 9):
    r = qft.run(10, 1024, nm, approximation_degree=d, seed=11)
    f = qft.approximation_fidelity(10, approximation_degree=d)
    print(f"{d:2d}  {r.circuit.count_ops().get('cp', 0):8d}  {f:21.3f}  {r.score:.3f}")

**Reading the table**

- A small d removes gates, raises the score under noise and keeps the output almost unchanged.
- A large d scores even higher, but the circuit is no longer a QFT. At d=9 there are no CP gates left.
- So compare approximate and exact QFT using **both** columns. A higher round-trip score alone does not make an approximation better.

## 10. QASM export

In [ ]:
print(qasm2.dumps(qft.build(3)))